# Kabsch Algorithm: Rigid Body Rotation Estimation

This notebook implements the **Kabsch algorithm** to find the optimal rotation matrix that maps a set of 3D points at time `t` to their positions at time `t+1`.

From the rotation matrix we extract:
- **Roll** (rotation about X-axis)
- **Pitch** (rotation about Y-axis)
- **Yaw** (rotation about Z-axis)

We simulate a **3×2 grid of 6 points** lying on a rigid body, apply a known translation + rotation, then recover the rotation via Kabsch.

---
### Theory: Kabsch Algorithm

Given point sets **P** (source) and **Q** (target), both centred:

1. Compute the cross-covariance matrix: `H = Pᵀ Q`
2. SVD: `H = U Σ Vᵀ`
3. Handle reflections: `d = sign(det(V Uᵀ))`
4. Optimal rotation: `R = V · diag(1, 1, d) · Uᵀ`
5. Translation: `t = centroid_Q − R · centroid_P`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.patches as mpatches
from scipy.spatial.transform import Rotation as ScipyRotation

np.set_printoptions(precision=4, suppress=True)
print('Libraries loaded.')

## 1. Generate the 3×2 Grid of Points (Time Step t)

In [ ]:
def make_grid_points(rows=3, cols=2, spacing=1.0):
    """Generate a rows×cols grid of 3D points in the XY plane."""
    pts = []
    for i in range(rows):
        for j in range(cols):
            pts.append([i * spacing, j * spacing, 0.0])
    return np.array(pts, dtype=float)

# Source points at time t
P = make_grid_points(rows=3, cols=2, spacing=1.0)

print(f'Source points P (shape {P.shape}):')
for i, p in enumerate(P):
    print(f'  p{i}: {p}')

## 2. Apply a Known Rotation + Translation to Get Time Step t+1

We define a ground-truth rotation using Euler angles (roll, pitch, yaw) and a translation vector.

In [ ]:
# ── Ground-truth transform ──────────────────────────────────────────────────
GT_ROLL_DEG  = 30.0   # rotation about X
GT_PITCH_DEG = 20.0   # rotation about Y
GT_YAW_DEG   = 45.0   # rotation about Z
GT_TRANSLATION = np.array([3.0, -1.5, 2.0])
# ───────────────────────────────────────────────────────────────────────────

def euler_to_rotation_matrix(roll_deg, pitch_deg, yaw_deg):
    """Intrinsic XYZ Euler angles → 3×3 rotation matrix."""
    r = np.radians(roll_deg)
    p = np.radians(pitch_deg)
    y = np.radians(yaw_deg)

    Rx = np.array([[1,       0,        0      ],
                   [0,  np.cos(r), -np.sin(r) ],
                   [0,  np.sin(r),  np.cos(r) ]])

    Ry = np.array([[ np.cos(p), 0, np.sin(p)],
                   [0,          1, 0         ],
                   [-np.sin(p), 0, np.cos(p) ]])

    Rz = np.array([[np.cos(y), -np.sin(y), 0],
                   [np.sin(y),  np.cos(y), 0],
                   [0,          0,          1]])

    return Rz @ Ry @ Rx   # intrinsic XYZ = extrinsic ZYX

R_gt = euler_to_rotation_matrix(GT_ROLL_DEG, GT_PITCH_DEG, GT_YAW_DEG)

# Apply transform: Q = R_gt @ P^T  then add translation
Q = (R_gt @ P.T).T + GT_TRANSLATION

print('Ground-truth rotation matrix R_gt:')
print(R_gt)
print(f'\nGround-truth translation: {GT_TRANSLATION}')
print(f'\nTarget points Q (shape {Q.shape}):')
for i, q in enumerate(Q):
    print(f'  q{i}: {q}')

## 3. Kabsch Algorithm Implementation

In [ ]:
def kabsch(P, Q):
    """
    Kabsch algorithm: find the optimal rotation R and translation t
    that minimises RMSD between P (source) and Q (target).

    Parameters
    ----------
    P : (N, 3) array  – source points at time t
    Q : (N, 3) array  – target points at time t+1

    Returns
    -------
    R : (3, 3) rotation matrix  (Q ≈ R @ P^T  + t, column-wise)
    t : (3,)   translation vector
    P_aligned : (N, 3) P after applying R and t
    rmsd : float  root-mean-square deviation after alignment
    """
    assert P.shape == Q.shape, 'P and Q must have the same shape'

    # Step 1: compute centroids and centre the point clouds
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    P_c = P - centroid_P
    Q_c = Q - centroid_Q

    # Step 2: cross-covariance matrix
    H = P_c.T @ Q_c   # shape (3, 3)

    # Step 3: SVD
    U, S, Vt = np.linalg.svd(H)
    V = Vt.T

    # Step 4: handle improper rotations (reflections)
    d = np.linalg.det(V @ U.T)
    D = np.diag([1.0, 1.0, np.sign(d)])

    # Step 5: optimal rotation
    R = V @ D @ U.T

    # Step 6: translation
    t = centroid_Q - R @ centroid_P

    # Apply alignment
    P_aligned = (R @ P.T).T + t

    # RMSD
    diff = P_aligned - Q
    rmsd = np.sqrt((diff ** 2).sum(axis=1).mean())

    return R, t, P_aligned, rmsd


R_est, t_est, P_aligned, rmsd = kabsch(P, Q)

print('Estimated rotation matrix R_est:')
print(R_est)
print(f'\nEstimated translation: {t_est}')
print(f'\nAlignment RMSD: {rmsd:.2e}  (should be ~0 for noise-free data)')

## 4. Extract Roll / Pitch / Yaw from the Rotation Matrix

In [ ]:
def rotation_matrix_to_euler(R, convention='ZYX'):
    """
    Convert a 3×3 rotation matrix to Euler angles.

    convention : 'ZYX'  → extrinsic ZYX = intrinsic XYZ
                         returns (roll, pitch, yaw) in degrees

    Uses scipy for robustness (handles gimbal lock gracefully).
    """
    r = ScipyRotation.from_matrix(R)
    # scipy extrinsic 'xyz' = R = Rz @ Ry @ Rx  (same as our ground truth)
    angles = r.as_euler('xyz', degrees=True)   # [roll, pitch, yaw]
    return angles  # (roll_x, pitch_y, yaw_z)


angles_gt  = rotation_matrix_to_euler(R_gt)
angles_est = rotation_matrix_to_euler(R_est)

labels = ['Roll (X) °', 'Pitch (Y) °', 'Yaw (Z) °']
print(f'{'Angle':<14}  {'Ground Truth':>14}  {'Estimated':>12}  {'Error':>10}')
print('-' * 55)
for lbl, gt, est in zip(labels, angles_gt, angles_est):
    print(f'{lbl:<14}  {gt:>14.4f}  {est:>12.4f}  {abs(gt-est):>10.2e}')

## 5. Visualisation: Points + Body Axes Before and After

In [ ]:
def draw_axes(ax, origin, R, length=0.8, alpha=1.0, linestyle='-', lw=2):
    """
    Draw the three body-frame axes (columns of R) from `origin`.
    X→red, Y→green, Z→blue.
    """
    colors = ['red', 'limegreen', 'royalblue']
    labels = ['X', 'Y', 'Z']
    for i in range(3):
        axis = R[:, i] * length
        ax.quiver(*origin, *axis,
                  color=colors[i], alpha=alpha,
                  linewidth=lw, linestyle=linestyle,
                  arrow_length_ratio=0.2)


def draw_rigid_body_edges(ax, pts, color, alpha=0.3):
    """Draw lines connecting the 3×2 grid neighbours."""
    # grid connectivity for 3 rows × 2 cols
    rows, cols = 3, 2
    for r in range(rows):
        for c in range(cols):
            idx = r * cols + c
            if c + 1 < cols:
                nb = r * cols + (c + 1)
                seg = np.array([pts[idx], pts[nb]])
                ax.plot(*seg.T, color=color, alpha=alpha, lw=1.5)
            if r + 1 < rows:
                nb = (r + 1) * cols + c
                seg = np.array([pts[idx], pts[nb]])
                ax.plot(*seg.T, color=color, alpha=alpha, lw=1.5)


# ── Body-frame axes: centred at the centroid of each point cloud ─────────────
centroid_P = P.mean(axis=0)
centroid_Q = Q.mean(axis=0)

I3 = np.eye(3)   # identity → original frame

# ── Figure ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 6))

# ── Left panel: 3D scatter with axes ─────────────────────────────────────────
ax1 = fig.add_subplot(121, projection='3d')

ax1.scatter(*P.T, c='steelblue', s=80, zorder=5, label='P  (time t)')
ax1.scatter(*Q.T, c='darkorange', s=80, zorder=5, marker='^', label='Q  (time t+1)')

draw_rigid_body_edges(ax1, P, color='steelblue')
draw_rigid_body_edges(ax1, Q, color='darkorange')

# Body-frame axes at source centroid (identity = world frame)
draw_axes(ax1, centroid_P, I3,    length=0.9, alpha=0.9, linestyle='--', lw=1.5)
# Body-frame axes at target centroid (rotated frame)
draw_axes(ax1, centroid_Q, R_est, length=0.9, alpha=1.0, linestyle='-',  lw=2.5)

# Axis-colour legend patches
ax1.text(*(centroid_P + np.array([-0.6, 0.1, 0.9])), 't  frame', fontsize=8,
         color='gray', style='italic')
ax1.text(*(centroid_Q + np.array([-0.6, 0.1, 0.9])), 't+1 frame', fontsize=8,
         color='gray', style='italic')

ax1.set_title('Rigid-Body Motion: Points & Body Axes', fontsize=11, fontweight='bold')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.legend(loc='upper left', fontsize=8)

# Axis colour guide (text box)
info = 'Axis colours:\n  Red   = X\n  Green = Y\n  Blue  = Z\nDashed = t frame\nSolid  = t+1 frame'
ax1.text2D(0.01, 0.01, info, transform=ax1.transAxes,
           fontsize=7.5, verticalalignment='bottom',
           bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# ── Right panel: bar chart of roll / pitch / yaw ──────────────────────────────
ax2 = fig.add_subplot(122)

angle_names = ['Roll\n(X)', 'Pitch\n(Y)', 'Yaw\n(Z)']
x = np.arange(3)
w = 0.32

bars_gt  = ax2.bar(x - w/2, angles_gt,  width=w, color='steelblue',  label='Ground truth',  alpha=0.85)
bars_est = ax2.bar(x + w/2, angles_est, width=w, color='darkorange', label='Kabsch estimate', alpha=0.85)

for bar in bars_gt:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h:.1f}°',
             ha='center', va='bottom', fontsize=8, color='steelblue')
for bar in bars_est:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h:.1f}°',
             ha='center', va='bottom', fontsize=8, color='darkorange')

ax2.set_xticks(x)
ax2.set_xticklabels(angle_names, fontsize=10)
ax2.set_ylabel('Angle (degrees)')
ax2.set_title('Recovered Roll / Pitch / Yaw\n(Ground Truth vs Kabsch)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.axhline(0, color='black', lw=0.8, linestyle='--')
ax2.set_ylim(-10, max(angles_gt) * 1.25 + 10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('kabsch_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as kabsch_result.png')

## 6. Sanity Checks

In [ ]:
print('=== Sanity Checks ===')
print(f'\n1. RMSD after alignment : {rmsd:.2e}   (expect ~0 for noise-free data)')

# Rotation matrix must be orthogonal: R^T R = I
ortho_err = np.linalg.norm(R_est.T @ R_est - np.eye(3))
print(f'2. Orthogonality error  : {ortho_err:.2e}   (expect ~0)')

# Determinant must be +1 (proper rotation, not reflection)
det = np.linalg.det(R_est)
print(f'3. det(R)               : {det:.6f}   (expect +1.0)')

# Compare estimated vs ground-truth rotation matrix element-wise
R_diff = np.linalg.norm(R_est - R_gt)
print(f'4. ||R_est - R_gt||_F   : {R_diff:.2e}   (expect ~0)')

# Pairwise distances preserved (rigidity check)
from itertools import combinations
dist_diffs = []
for i, j in combinations(range(len(P)), 2):
    d_P = np.linalg.norm(P[i] - P[j])
    d_Q = np.linalg.norm(Q[i] - Q[j])
    dist_diffs.append(abs(d_P - d_Q))
print(f'5. Max pairwise dist change: {max(dist_diffs):.2e}   (rigidity check, expect ~0)')

## 7. Bonus: Robustness Test with Gaussian Noise

In practice, sensor measurements are noisy. Kabsch still finds the best least-squares rotation.

In [ ]:
noise_levels = [0.0, 0.02, 0.05, 0.10, 0.20, 0.50]
rmsds, roll_errs, pitch_errs, yaw_errs = [], [], [], []

np.random.seed(42)
for sigma in noise_levels:
    Q_noisy = Q + np.random.randn(*Q.shape) * sigma
    R_n, t_n, _, rm = kabsch(P, Q_noisy)
    angs = rotation_matrix_to_euler(R_n)
    rmsds.append(rm)
    roll_errs.append(abs(angs[0] - angles_gt[0]))
    pitch_errs.append(abs(angs[1] - angles_gt[1]))
    yaw_errs.append(abs(angs[2] - angles_gt[2]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(noise_levels, rmsds, 'o-', color='purple', lw=2)
axes[0].set_xlabel('Noise σ (metres)')
axes[0].set_ylabel('Alignment RMSD (metres)')
axes[0].set_title('RMSD vs Noise Level')
axes[0].grid(alpha=0.3)

axes[1].plot(noise_levels, roll_errs,  's-', color='red',       lw=2, label='Roll error')
axes[1].plot(noise_levels, pitch_errs, '^-', color='darkorange', lw=2, label='Pitch error')
axes[1].plot(noise_levels, yaw_errs,   'D-', color='royalblue', lw=2, label='Yaw error')
axes[1].set_xlabel('Noise σ (metres)')
axes[1].set_ylabel('Angle error (degrees)')
axes[1].set_title('Roll/Pitch/Yaw Error vs Noise')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Kabsch Robustness to Sensor Noise', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('kabsch_noise.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n{'Noise σ':>10} | {'RMSD':>10} | {'Roll err°':>10} | {'Pitch err°':>11} | {'Yaw err°':>10}')
print('-' * 60)
for s, r, re, pe, ye in zip(noise_levels, rmsds, roll_errs, pitch_errs, yaw_errs):
    print(f'{s:>10.2f} | {r:>10.4f} | {re:>10.4f} | {pe:>11.4f} | {ye:>10.4f}')